In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
!pip install rapidfuzz

In [ ]:
sheet = gc.open("Scholarship Tracker").sheet1
data = sheet.get_all_records()
print(f"{len(data)} rows loaded")

12 rows loaded


In [ ]:
from rapidfuzz import fuzz
from datetime import datetime

def generate_digest(data, similarity_threshold=80, days_ahead=7):
    names = [row['Scholarship Name'].strip() for row in data]
    today = datetime.now().date()

    print("-" * 50)
    print("SCHOLARSHIP TRACKER DIGEST")
    print("-" * 50)

    print(f"\nChecking {len(names)} entries for duplicates!\n")
    duplicates_found = 0

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            similarity = fuzz.ratio(names[i].lower(), names[j].lower())
            if similarity > similarity_threshold:
                duplicates_found += 1
                print(f"  Possible duplicate ({similarity:.0f}% similar):")
                print(f"      - {names[i]}")
                print(f"      - {names[j]}\n")

    if duplicates_found == 0:
        print("  No likely duplicates found.\n")

    print(f"Checking for deadlines within {days_ahead} days\n")
    urgent_found = 0

    for row in data:
        if not row['Deadline']:
            continue
        deadline = datetime.strptime(row['Deadline'], '%Y-%m-%d').date()
        days_left = (deadline - today).days
        if 0 <= days_left <= days_ahead:
            urgent_found += 1
            print(f"  {row['Scholarship Name'].strip()} closes in {days_left} day(s) (deadline: {row['Deadline']})")

    if urgent_found == 0:
        print("  No scholarships closing soon.\n")

    print("-" * 50)
    print(f"SUMMARY:\n- {len(data)} total entries\n- {duplicates_found} possible duplicate(s)\n- {urgent_found} urgent deadline(s)")
    print("-" * 50)


generate_digest(data)

--------------------------------------------------
SCHOLARSHIP TRACKER DIGEST
--------------------------------------------------

Checking 12 entries for duplicates!

  Possible duplicate (93% similar):
      - UCalgary Excellence Award
      - U of Calgary Excellence Award

  Possible duplicate (100% similar):
      - Alberta STEM Bursary
      - Alberta Stem Bursary

Checking for deadlines within 7 days

  U of Calgary Excellence Award closes in 6 day(s) (deadline: 2026-06-25)
  Schulich Leader Scholarship closes in 1 day(s) (deadline: 2026-06-20)
--------------------------------------------------
SUMMARY:
- 12 total entries
- 2 possible duplicate(s)
- 2 urgent deadline(s)
--------------------------------------------------


In [ ]:
import requests

url = "https://script.google.com/macros/s/AKfycbyy45WgLLUtUt-iJVw-lQBjzblgMQmsVaCpirULPRReOvOo4VamDpPX9gTMz3Tmqk5LQA/exec"

test_body = """Hi Alissa,
You have 3 matching scholarship(s) totalling $2,000 on ScholarshipsCanada.com.
The following scholarships have a deadline in the next 40 days.
SCHOLARSHIP ALERTS
Constable Don Doucet Memorial Scholarship Fund
Award Amount: $1,000
Deadline: May 15, 2026
click here for more information
Kentville Volunteer Fire Department Memorial Bursary
Award Amount: $1,000
Deadline: May 21, 2026
click here for more information
The Bernard Chernos Contest
Award Amount: n/s
Deadline: May 22, 2026
click here for more information"""

response = requests.post(url, json={"body": test_body, "date": "2026-05-05"})
print(response.status_code)
print(response.text)

200
{"status":"success","added":0,"skipped":3}
